# 따라 하기 — 크롤링 연습 사이트 quotes.toscrape.com
크롤링 전 확인: robots.txt · 이용약관 · 공공누리 표시 · 서버 예절(1초 쉬기) · **모은 데이터는 온라인에 올리지 않기**

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

## ① 한 페이지에서 모으기

In [ ]:
import requests, time, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
BASE = 'https://quotes.toscrape.com'
soup = BeautifulSoup(requests.get(BASE).text, 'html.parser')
rows = []
for box in soup.select('div.quote'):
    rows.append({
        '명언': box.select_one('span.text').get_text(strip=True),
        '저자': box.select_one('small.author').get_text(strip=True),
        '링크': urljoin(BASE, box.select_one('a')['href']),
    })
pd.DataFrame(rows).head()

## ② 목록에서 상세 페이지로

In [ ]:
for r in rows[:3]:                                # 먼저 3개만 시험
    d = BeautifulSoup(requests.get(r['링크']).text, 'html.parser')
    r['태어난 날'] = d.select_one('span.author-born-date').get_text(strip=True)
    r['소개'] = d.select_one('div.author-description').get_text(' ', strip=True)
    time.sleep(1)                                 # 서버에 부담 주지 않기
pd.DataFrame(rows[:3])

## ③ 여러 페이지 돌고 CSV로 저장

In [ ]:
rows = []
for page in range(1, 6):                # 1~5페이지
    url  = f'{BASE}/page/{page}/'
    soup = BeautifulSoup(requests.get(url).text, 'html.parser')
    for box in soup.select('div.quote'):
        rows.append({'페이지': page,
                     '저자': box.select_one('small.author').get_text(strip=True)})
    time.sleep(1)

df = pd.DataFrame(rows)
df.to_csv('quotes.csv', index=False, encoding='utf-8-sig')
df['저자'].value_counts().head()